In [ ]:
import os, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import cv2
from glob import glob

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models


import albumentations as A
from albumentations.pytorch import ToTensorV2

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
IMG_SIZE = 224

# Contrast enhancement using CLAHE
def enhance_mri(img):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return clahe.apply(img)

# Augmentations
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.3),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2()
])

# Dataset Class
class BrainMRIDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None, enhance=True):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform
        self.enhance = enhance
        
    def __len__(self):
        return len(self.filepaths)
        
    def __getitem__(self, idx):
        p = self.filepaths[idx]
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise RuntimeError("Failed to read image: "+p)
            
        if self.enhance:
            img = enhance_mri(img)
        
        augmented = self.transform(image=img) if self.transform else {"image":img}
        img_t = augmented["image"]
        
        # ensure single channel tensor (1, H, W)
        if img_t.shape[0] != 1:
            img_t = img_t.mean(dim=0, keepdim=True)
            
        label = self.labels[idx]
        return img_t.float(), label

# Load Entire Dataset (Training + Testing)
base_data_path = "/kaggle/input/brain-tumor-mri-dataset"

filepaths, labels = [], []
le = LabelEncoder()


data_subdirs = [f.path for f in os.scandir(base_data_path) if f.is_dir()]

for data_dir in data_subdirs:
    class_folders = [f.name for f in os.scandir(data_dir) if f.is_dir()]
    for cls_name in class_folders:
        full_class_path = os.path.join(data_dir, cls_name)
        fpaths = glob(os.path.join(full_class_path, "*"))
        fpaths = [p for p in fpaths if os.path.splitext(p)[1].lower() in
                  [".jpg",".jpeg",".png",".bmp"]]
        filepaths += fpaths
        labels += [cls_name]*len(fpaths)

if len(filepaths) == 0:
    raise RuntimeError("No images found — check dataset structure.")

labels_encoded = le.fit_transform(labels)
print("Label classes:", le.classes_)
print(f"Total images loaded from all directories: {len(filepaths)}")


In [ ]:
# Dataset Splitting
NUM_CLIENTS = 5
TEST_SPLIT = 0.1
VAL_SPLIT = 0.1

indices = np.arange(len(filepaths))

# Stratified Test Split
train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SPLIT,
    stratify=labels_encoded,
    random_state=SEED
)

# Stratified Validation Split
val_fraction = VAL_SPLIT / (1 - TEST_SPLIT)
train_idx, val_idx = train_test_split(
    train_idx,
    test_size=val_fraction,
    stratify=labels_encoded[train_idx],
    random_state=SEED
)

# Split training data across clients
client_indices = [[] for _ in range(NUM_CLIENTS)]

for i, idx in enumerate(train_idx):
    client_indices[i % NUM_CLIENTS].append(int(idx))

for cid, cidx in enumerate(client_indices):
    if len(cidx) > 0:
        classes, counts = np.unique(labels_encoded[cidx], return_counts=True)
        class_dist = dict(zip(le.classes_[classes.astype(int)], counts))
    else:
        class_dist = {}

    print(f"Client {cid}: samples={len(cidx)}, class_counts={class_dist}")


In [ ]:
# Dataloader Factory
def get_dataloader_from_indices(idxs, batch_size=16, train=True):
    paths = [filepaths[i] for i in idxs]
    labs = labels_encoded[idxs]

    transform = train_transform if train else val_transform

    dataset = BrainMRIDataset(
        filepaths=paths,
        labels=labs,
        transform=transform,
        enhance=True
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=train,
        num_workers=2,
        pin_memory=True
    )

test_loader = get_dataloader_from_indices(test_idx, batch_size=32, train=False)
val_loader  = get_dataloader_from_indices(val_idx,  batch_size=32, train=False)

In [ ]:
class MobileNetV2Model(nn.Module):
    def __init__(self, num_classes=4, input_ch=1, feat_dim=1280):
        super().__init__()

        # Load pretrained MobileNetV2 backbone
        backbone = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

        # Modify first conv to support grayscale
        first_conv = backbone.features[0][0]
        backbone.features[0][0] = nn.Conv2d(
            in_channels=input_ch,
            out_channels=first_conv.out_channels,
            kernel_size=first_conv.kernel_size,
            stride=first_conv.stride,
            padding=first_conv.padding,
            bias=first_conv.bias
        )

        # Remove original classifier
        backbone.classifier = nn.Identity()

        self.backbone = backbone

        # Custom feature + classifier layers
        self.fc = nn.Linear(feat_dim, feat_dim)
        self.classifier = nn.Linear(feat_dim, num_classes)

    def forward(self, x, return_features=False):
        x = self.backbone.features(x)
        x = x.mean([2, 3]) # Global average pooling

        feat = self.fc(x)

        if return_features:
            return feat

        return self.classifier(feat)

#Build Model
model = MobileNetV2Model(num_classes=len(le.classes_), input_ch=1).to(DEVICE)
print("Total parameters:", sum(p.numel() for p in model.parameters()))


In [ ]:
class LocalMetaLearners:
    def __init__(self, feature_dim=1280, num_classes=4):
        self.feature_dim = feature_dim
        self.num_classes = num_classes

        #MLP classifier
        self.mlp = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        ).to(DEVICE)

        #SVM components
        self.svm = None
        self.scaler = StandardScaler()

        # ELM parameters
        self.elm_W = None
        self.elm_bias = None

    # Train MLP, SVM, and ELM on local extracted features
    def train_all(self, features_np, labels_np, n_epochs=30, batch_size=32, lr=1e-3):

        #Train MLP
        X = torch.from_numpy(features_np).float().to(DEVICE)
        y = torch.from_numpy(labels_np).long().to(DEVICE)

        dl = torch.utils.data.DataLoader(
            torch.utils.data.TensorDataset(X, y),
            batch_size=batch_size,
            shuffle=True
        )

        optimizer = torch.optim.Adam(self.mlp.parameters(), lr=lr)
        loss_fn = nn.CrossEntropyLoss()

        self.mlp.train()
        for epoch in range(n_epochs):
            for xb, yb in dl:
                optimizer.zero_grad()
                loss = loss_fn(self.mlp(xb), yb)
                loss.backward()
                optimizer.step()

        #Train SVM 
        Xs = self.scaler.fit_transform(features_np)
        self.svm = SVC(kernel='rbf', probability=True)
        self.svm.fit(Xs, labels_np)

        #Train ELM (ridge regression)
        H = np.hstack([features_np, np.ones((features_np.shape[0], 1))])
        T = np.eye(self.num_classes)[labels_np]
        lam = 1e-3

        A = H.T @ H + lam * np.eye(H.shape[1])
        B = H.T @ T
        W = np.linalg.solve(A, B)

        self.elm_W = W[:-1]
        self.elm_bias = W[-1]

    # Predict (ensemble averaging)
    def predict_proba(self, features_np):
        #MLP
        with torch.no_grad():
            logits = self.mlp(torch.from_numpy(features_np).float().to(DEVICE))
            probs_mlp = torch.softmax(logits, dim=1).cpu().numpy()

        #SVM
        probs_svm = self.svm.predict_proba(self.scaler.transform(features_np))

        #ELM
        logits_elm = features_np @ self.elm_W + self.elm_bias
        exp = np.exp(logits_elm - logits_elm.max(axis=1, keepdims=True))
        probs_elm = exp / exp.sum(axis=1, keepdims=True)

        return (probs_mlp + probs_svm + probs_elm) / 3

In [ ]:
# FedProx Loss
def fedprox_loss(global_model, local_model, mu):
    prox = 0.0
    for (_, local_param), (_, global_param) in zip(local_model.named_parameters(),
                                                   global_model.named_parameters()):
        diff = local_param - global_param.to(local_param.device)
        prox += diff.norm(2).pow(2)
    return (mu / 2.0) * prox

# Model weight utilities
def get_model_weights(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def set_model_weights(model, weight_dict):
    state = {
        name: weight_dict[name].to(model.device)
        if hasattr(model, "device") else weight_dict[name]
        for name in weight_dict
    }
    model.load_state_dict(state)

In [ ]:
class FLClient:
    def __init__(self, client_id, train_indices, backbone_template,
                 local_epochs=2, batch_size=16, lr=5e-5, mu=0.001):

        self.id = client_id
        self.train_indices = train_indices
        self.local_epochs = local_epochs
        self.lr = lr
        self.mu = mu

        # Local dataloader
        self.dl = get_dataloader_from_indices(train_indices, batch_size=batch_size, train=True)

        # Local model copy
        self.model = MobileNetV2Model(num_classes=len(le.classes_), input_ch=1).to(DEVICE)
        self.backbone_template = backbone_template

        # Compute class weights
        local_labels = labels_encoded[train_indices]
        if len(local_labels) == 0:
            self.class_weights = torch.ones(len(le.classes_)).to(DEVICE)
        else:
            weights = class_weight.compute_class_weight(
                "balanced",
                classes=np.arange(len(le.classes_)),
                y=local_labels
            )
            self.class_weights = torch.tensor(weights).float().to(DEVICE)

        # Local meta learners
        self.meta = LocalMetaLearners(feature_dim=1280, num_classes=len(le.classes_))

   
    def set_backbone(self, state_dict):
        self.model.load_state_dict(state_dict)


    def local_train(self, global_model_for_prox=None):
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights)

        self.model.train()
        for _ in range(self.local_epochs):
            for xb, yb in self.dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = self.model(xb)

                loss = loss_fn(logits, yb)
                if global_model_for_prox and self.mu > 0:
                    loss += fedprox_loss(global_model_for_prox, self.model, self.mu)

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()

        # Extract features
        self.model.eval()
        feats, labs = [], []
        with torch.no_grad():
            loader = get_dataloader_from_indices(self.train_indices, batch_size=64, train=False)
            for xb, yb in loader:
                f = self.model(xb.to(DEVICE), return_features=True).cpu().numpy()
                feats.append(f)
                labs.append(yb.numpy())

        feats = np.vstack(feats)
        labs = np.concatenate(labs)

        # Train meta-learners
        self.meta.train_all(feats, labs, n_epochs=20, batch_size=64, lr=1e-3)

        # Return updated weights
        return get_model_weights(self.model), len(self.train_indices)

In [ ]:
# FedAvg Aggregation
def aggregate_weights(weight_sample_pairs):
    total_samples = sum(ns for _, ns in weight_sample_pairs)

    if total_samples == 0:
        return {k: v.clone() for k, v in weight_sample_pairs[0][0].items()} if weight_sample_pairs else {}

    agg = {}
    first_weights = weight_sample_pairs[0][0]

    for k in first_weights.keys():
        agg[k] = sum(w[k].float() * ns for w, ns in weight_sample_pairs) / total_samples

    return agg

# Initialize Global Model
global_model = MobileNetV2Model(num_classes=len(le.classes_), input_ch=1).to(DEVICE)
global_weights = get_model_weights(global_model)


# Create Clients
clients = []
for i in range(NUM_CLIENTS):
    client = FLClient(
        client_id=i,
        train_indices=client_indices[i],
        backbone_template=global_model,
        local_epochs=2,
        batch_size=16,
        lr=1e-3,
        mu=0.0001
    )
    client.set_backbone(global_weights)
    clients.append(client)


ROUNDS = 20
metrics_history = []

# Federated Training Loop
for rnd in range(1, ROUNDS + 1):
    print(f"\n--- Round {rnd}/{ROUNDS} ---")

    # Sync clients with global weights
    for client in clients:
        client.set_backbone(global_weights)

    # Local training
    client_updates = []
    for client in clients:
        print(f"Client {client.id} training...")
        updated_w, num_samples = client.local_train(global_model_for_prox=global_model)
        client_updates.append((updated_w, num_samples))

    # FedAvg aggregation
    global_weights = aggregate_weights(client_updates)
    global_model.load_state_dict({k: v.to(DEVICE) for k, v in global_weights.items()})

    # Validation
    global_model.eval()
    preds, gts = [], []

    with torch.no_grad():
        for xb, yb in val_loader:
            out = global_model(xb.to(DEVICE))
            preds.extend(out.argmax(dim=1).cpu().tolist())
            gts.extend(yb.tolist())

    acc = accuracy_score(gts, preds)
    report = classification_report(gts, preds, target_names=le.classes_, output_dict=True)

    # Store metrics
    metrics_history.append({
        "round": rnd,
        "accuracy": acc,
        **{f"{avg}_precision": report[avg]["precision"] for avg in ["macro avg", "weighted avg"]},
        **{f"{avg}_recall": report[avg]["recall"] for avg in ["macro avg", "weighted avg"]},
        **{f"{avg}_f1": report[avg]["f1-score"] for avg in ["macro avg", "weighted avg"]},
        **{f"{cls}_f1": report[cls]["f1-score"] for cls in le.classes_}
    })

    print(f"Round {rnd} Accuracy: {acc:.4f} | Macro F1: {report['macro avg']['f1-score']:.4f}")

In [ ]:
metrics_df = pd.DataFrame(metrics_history)
print("\nFederated Training Complete.\nMetrics:")
print(metrics_df)

In [ ]:
global_model.eval()

test_features = []
test_gt = []

with torch.no_grad():
    for xb, yb in test_loader:
        feats = global_model(xb.to(DEVICE), return_features=True)
        test_features.append(feats.cpu().numpy())
        test_gt.append(yb.numpy())

test_features = np.vstack(test_features)
test_gt = np.concatenate(test_gt)

# Ensemble predictions from all clients' meta-learners
num_classes = len(le.classes_)
probs_sum = np.zeros((len(test_features), num_classes))

for client in clients:
    probs_sum += client.meta.predict_proba(test_features)

probs_avg = probs_sum / len(clients)
y_pred = probs_avg.argmax(axis=1)

# Results
print("Global Backbone + Local Meta-Learner Ensemble Accuracy:",
      accuracy_score(test_gt, y_pred))
print("Final Macro F1-score:", f1_score(test_gt, y_pred, average="macro"))
print("Final Weighted F1-score:", f1_score(test_gt, y_pred, average="weighted"))
print(classification_report(test_gt, y_pred,
                            target_names=le.classes_, digits=6))


In [ ]:
save_path = "/kaggle/working/mobilenetv2_global_backbone.pth"

os.makedirs("/kaggle/working/", exist_ok=True)
torch.save(global_model.state_dict(), save_path)

print(f"Saved global MobileNetV2 backbone to: {save_path}")

In [ ]:
path = filepaths[0]
original = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

if original is None:
    raise RuntimeError(f"Failed to load: {path}")

# CLAHE enhancement
enhanced = enhance_mri(original)

# Apply validation transform
trans_out = val_transform(image=enhanced)["image"]

# Denormalize for display
denorm = (trans_out * 0.5 + 0.5).squeeze().cpu().numpy()
denorm_disp = (denorm * 255).astype(np.uint8)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(original, cmap='gray');     axes[0].set_title("Original"); axes[0].axis('off')
axes[1].imshow(enhanced, cmap='gray');     axes[1].set_title("Enhanced (CLAHE)"); axes[1].axis('off')
axes[2].imshow(denorm_disp, cmap='gray');  axes[2].set_title("Transformed (Denormalized)"); axes[2].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
#Confusion Matrix
cm = confusion_matrix(test_labels, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=le.classes_,
    yticklabels=le.classes_
)

plt.title("Confusion Matrix — Final Global + Meta-Ensemble Model")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

print("Generated confusion matrix for the final model.")


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(metrics_df['round'], metrics_df['accuracy'], label='Accuracy', marker='o', color='blue')
plt.title('Global Model Validation Accuracy Over Rounds')
plt.xlabel('Federated Learning Round')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.xticks(metrics_df['round'])
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(metrics_df['round'], metrics_df['macro avg_precision'], label='Macro Avg Precision', marker='x', color='green')
plt.title('Global Model Validation Macro Avg Precision Over Rounds')
plt.xlabel('Federated Learning Round')
plt.ylabel('Macro Avg Precision')
plt.legend()
plt.grid(True)
plt.xticks(metrics_df['round'])
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(metrics_df['round'], metrics_df['macro avg_recall'], label='Macro Avg Recall', marker='s', color='red')
plt.title('Global Model Validation Macro Avg Recall Over Rounds')
plt.xlabel('Federated Learning Round')
plt.ylabel('Macro Avg Recall')
plt.legend()
plt.grid(True)
plt.xticks(metrics_df['round'])
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(metrics_df['round'], metrics_df['macro avg_f1'], label='Macro Avg F1-score', marker='D', color='purple')
plt.title('Global Model Validation Macro Avg F1-score Over Rounds')
plt.xlabel('Federated Learning Round')
plt.ylabel('Macro Avg F1-score')
plt.legend()
plt.grid(True)
plt.xticks(metrics_df['round'])
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 8))

plt.plot(metrics_df['round'], metrics_df['accuracy'], label='Accuracy', marker='o')
plt.plot(metrics_df['round'], metrics_df['macro avg_precision'], label='Macro Avg Precision', marker='x')
plt.plot(metrics_df['round'], metrics_df['macro avg_recall'], label='Macro Avg Recall', marker='s')
plt.plot(metrics_df['round'], metrics_df['macro avg_f1'], label='Macro Avg F1-score', marker='D')

plt.title('Global Model Validation Metrics Over Rounds')
plt.xlabel('Federated Learning Round')
plt.ylabel('Metric Value')
plt.legend()
plt.grid(True)
plt.xticks(metrics_df['round'])
plt.tight_layout()
plt.show()

print("Generated line plots for global model's validation metrics.")